In [72]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init
from torch import autograd as Grad

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
import matplotlib

import ipywidgets as widgets
from IPython.display import display
import trimesh

import random
import math
#import pyrender

import time
import os

import open3d as o3d
print('Using open3d version',o3d.__version__)


from blend_utils import *
from mesh_processing import *

import differential
import importlib
importlib.reload(differential)
from differential import *

two_pi = 2*torch.pi
diffmod = DifferentialModule()

# Set device priority: MPS > CUDA > CPU

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


#device = torch.device('cpu')
print(f"Using device: {device}")


Using open3d version 0.18.0
Using device: mps


In [ ]:
shapename = 'igea11706_6patch' #'star5_4patch'#'igea11706_6patch'

'''
def posenc(x):
    p = 0.2 * x  # (N, 2)
    freqs = 2 ** torch.arange(0, 8, device=x.device).float()  # [1, 2, 4, 8, 16, 32, 64, 128]
    p = p.unsqueeze(-1) * freqs * two_pi  # (N, 2, 8)

    encoded = torch.cat([torch.sin(p), torch.cos(p)], dim=-1)  # (N, 2, 16)
    return encoded.view(x.shape[0], -1)  # Flatten to (N, 32)
'''


def posenc(x):
    p = 0.2 * x  # (N, 2)
    freqs = 2 ** torch.arange(0, 8, device=x.device).float()  # [1, 2, ..., 64, 128]
    p0 = p[:, 0].unsqueeze(1) * freqs * two_pi  # (N, 8)
    p1 = p[:, 1].unsqueeze(1) * freqs * two_pi  # (N, 8)

    sincos = torch.stack([
        torch.sin(p0), torch.sin(p1),
        torch.cos(p0), torch.cos(p1)
    ], dim=2)  # (N, 8, 4)

    return sincos.view(x.shape[0], -1)  # (N, 32)
    
layer_sizes = [32, 64, 32, 3]


In [ ]:
def B(t):
    return pou_blend_func(t, v=0.6)

class BNS(nn.Module):
    def __init__(self, layer_sizes, he_mesh=None, from_file = False, shapename=None, device=None):
        super(BNS, self).__init__()

        self.device = device

        if from_file==False:
            self.he_mesh=he_mesh
            self.patches_path = None
        else:
            self.patches_path = 'patches/'+shapename
            filepath = self.patches_path+'/coarse.obj'
            coarse_mesh_o3d = o3d.io.read_triangle_mesh(filepath)
            self.he_mesh = o3d.geometry.HalfEdgeTriangleMesh.create_from_triangle_mesh(coarse_mesh_o3d)


        #halfedge stuff for the coarse representation
        self.V = torch.tensor(np.asarray(self.he_mesh.vertices), dtype=torch.float32, device=self.device)
        self.n = self.V.shape[0]
        self.F = np.asarray(self.he_mesh.triangles)
        self.he = self.he_mesh.half_edges

        self.num_patches = self.F.shape[0]

        # a halfedge is assigned to each vertex (onering)
        print('Assiging a halfedge to each vertex.')
        initial_V_he = get_V_he(self.he_mesh)
        
        self.V_he = optimise_V_he(self.he_mesh, initial_V_he)

        self.base_triangle_verts = torch.tensor([[0.0, 0.0],
                      [1.0, 0.0],
                      [0.5, np.sqrt(3)/2]], device=self.device)


        print('Computing onering data.')

        self.onerings = compute_onering_data(self.he_mesh, self.V_he)
        for onering in self.onerings:
            print(onering )
        
        self.mlps = nn.ModuleList([MLP(layer_sizes) for _ in range(self.V.shape[0])])

        print('Computing rotations.')
        self.compute_fixed_rotations()

        self.default_coarse_points = self.V
        self.default_rotations = self.rotations



    def reset(self):
        self.rotations = self.default_rotations
        self.coarse_points = self.default_coarse_points
        self.weights = [1 for i in range(self.n)]

    def compute_fixed_rotations(self):


        triangle_mesh = o3d.geometry.TriangleMesh()
        triangle_mesh.vertices = self.he_mesh.vertices
        triangle_mesh.triangles = self.he_mesh.triangles
        triangle_mesh.compute_vertex_normals()
        
        self.rotations = []
        for i in range(self.n):
            normal = torch.tensor( triangle_mesh.vertex_normals[i] ).float()
            v = self.he[self.V_he[i]].vertex_indices[1] #the chosen neighbour vertex
            dir_0 = ( self.V[v,:] - self.V[i,:] ).float()
            dir_0 = dir_0 - (dir_0*normal).sum()*normal
            dir_0 = dir_0 / torch.sqrt((dir_0*dir_0).sum())

            R = torch.stack([normal, dir_0, torch.cross(normal, dir_0, dim=0)   ] ).transpose(1,0)

            self.rotations.append(R)

    '''
    def compute_samples_sphere(self, x):
        bary = barycentric_coordinates(x[:,0], x[:,1])
        
        embedded_triangles = []
        for triangle_index in range(self.F.shape[0]):
            verts = self.V[self.F[triangle_index,:]]
            

            
            embedded_triangles.append ( bary @ verts )
        embedded_triangles = torch.stack(embedded_triangles)

        norms = torch.norm(embedded_triangles, dim=-1, keepdim=True)

        target_shape = embedded_triangles / norms
        target_normals = target_shape
        
        return target_shape, target_normals, embedded_triangles, bary
    '''




    def compute_samples(self, num_samples=10000):
        if self.patches_path is None:
            raise TypeError("The BNS object has no patches filepath.")
    
        uv_samples = []
        target = []
        target_normals = []
    
        for i in range(self.F.shape[0]):
            filepath = os.path.join(self.patches_path, f"{i}_param.obj")
            mesh = o3d.io.read_triangle_mesh(filepath)
    
            V = torch.from_numpy(np.asarray(mesh.vertices)).float().to(self.device)
            F = torch.from_numpy(np.asarray(mesh.triangles)).long().to(self.device)
            UV = torch.from_numpy(np.asarray(mesh.triangle_uvs).reshape(-1, 3, 2)).float().to(self.device)
            mesh.compute_triangle_normals()
            N = torch.from_numpy(np.asarray(mesh.triangle_normals)).float().to(self.device)


            #print('normals', N.shape)
    
            # Compute 3D face areas
            v0, v1, v2 = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
            face_areas = 0.5 * torch.norm(torch.cross(v1 - v0, v2 - v0, dim=1), dim=1)
            probs = face_areas / face_areas.sum()
    
            # Sample face indices
            face_ids = torch.multinomial(probs, num_samples, replacement=True)
    
            # Barycentric sampling
            u = torch.rand(num_samples, 1).to(self.device)
            v = (torch.rand(num_samples, 1) * (1 - u) ).to(self.device)
            w = 1 - u - v
            bary = torch.cat([u, v, w], dim=1).unsqueeze(-1)  # shape (N, 3, 1)
    
            # Fetch triangle UVs and vertices
            tri_uvs = UV[face_ids]           # (N, 3, 2)
            tri_verts = V[F[face_ids]]       # (N, 3, 3)
            tri_normals = N[face_ids]
    
            # Barycentric interpolation via broadcasting
            face_uv_samples = (tri_uvs * bary).sum(dim=1)      # (N, 2)
            face_target = (tri_verts * bary).sum(dim=1)        # (N, 3)
    
            uv_samples.append(face_uv_samples.unsqueeze(0))
            target.append(face_target.unsqueeze(0))
            target_normals.append(tri_normals.unsqueeze(0))
    
        uv_samples = torch.cat(uv_samples, dim=0)
        target = torch.cat(target, dim=0)
        target_normals = torch.cat(target_normals, dim=0)

        x = uv_samples
        bary = barycentric_coordinates(x[:,:,0], x[:,:,1])


        #bary = barycentric_coordinates(x[:,0], x[:,1])
        #print(bary.shape)
        #plt.scatter(x[:,0], x[:,1], c=bary)
        #plt.show()
        embedded_triangles = []
        #for triangle_index in range(self.F.shape[0]):
        #    verts = self.V[self.F[triangle_index,:]]
            #print(verts)
        #    embedded_triangles.append ( bary @ verts )
        #embedded_triangles = torch.stack(embedded_triangles)

        print('uv, bary', uv_samples.shape, bary.shape)
    
        return uv_samples, target, target_normals, bary, embedded_triangles
        


    def onering_coords(self, x, i, j, angle, stretch):

        ### VEC: make j into J (length 80 vector), and angle and stretch also become length 80 vectors.
        ### these vectors do not change for the whole training process! just compute them at the start.
        ### x is the only thing that changes.
        ### we could precompute pt[I]
        ### we could precompute J*angle
        ### we could precompute I*2*torch.pi/3
    

        v = x-self.base_triangle_verts[i]
        r = torch.sqrt(v.pow(2).sum(-1))
        
        theta = torch.atan2(v[:,1], v[:,0]) 
    
        theta -= i*(two_pi/3) #make it the angle from the relevant edge
    
        theta = theta + (theta<0)*(two_pi) #make sure theta is in [0,2pi)

        #print(theta.shape, stretch_vec.shape, J.shape, angle_vec.shape)
        new_theta = stretch * -1*theta + (j*angle)
        #print(new_theta.shape)
       
    
        new_theta = new_theta + (new_theta<0)*(two_pi) #make sure theta is in [0,2pi)
        new_theta = new_theta - (new_theta>two_pi)*(two_pi) #make sure theta is in [0,2pi)
        
        onering_x = torch.stack([r*torch.cos(new_theta), r*torch.sin(new_theta)]).float().transpose(1,0)
        return onering_x, r, theta, new_theta

    def save_mlps(self, filename="mlps.pth"):
        torch.save({
            'mlps_state_dict': [mlp.state_dict() for mlp in self.mlps],
        }, filename)
    
    def load_mlps(self, filename="mlps.pth"):
        checkpoint = torch.load(filename)
        
        for mlp, state_dict in zip(self.mlps, checkpoint['mlps_state_dict']):
            mlp.load_state_dict(state_dict)
        
    
    def forward(self, x, zero_test=False, return_scalar_fields=False):
        
        
        f = torch.zeros((self.F.shape[0], x.shape[1], 3), device=self.device)
        if return_scalar_fields:
            scalar_field_for_testing = {'blend': np.zeros( f.shape  ), 
                                   'angle': -10*np.ones( (f.shape[0], f.shape[1]) )}
        
        for triangle_index in range(self.F.shape[0]):
            verts = self.F[triangle_index,:]

            #print('verts', verts)
            #print('onerings', len(self.onerings))

            virtual_vertex = ( self.V[verts[0]] + self.V[verts[1]] + self.V[verts[2]] )/3
            blendsum = torch.zeros((x.shape[1],1))
            
            for i in range(3):
            
                onering = self.onerings[verts[i]]
                #valence = onering['valence']
                #which face in onering matrix?
                
                j = onering['triangles'].index(triangle_index)
                #j = 0
                onering_x, r, theta, new_theta = self.onering_coords(x[triangle_index,:,:].squeeze(), i, j, onering['angle'], onering['angle_stretch'])

                if return_scalar_fields:
                    if verts[i]==0:
                        scalar_field_for_testing['blend'][triangle_index,:,0] += B(r).detach().numpy()
    
                    if verts[i]==1:
                        scalar_field_for_testing['blend'][triangle_index,:,1] += B(r).detach().numpy()
                        
                    if verts[i]==2:
                        scalar_field_for_testing['blend'][triangle_index,:,2] += B(r).detach().numpy()
    
                    scalar_field_for_testing['blend'][triangle_index,: , :][r<0.05] = 1 
                    scalar_field_for_testing['blend'][triangle_index,: , :][r<0.01] = 0 

                    if verts[i]==0:
                        #scalar_field_for_testing['angle'][triangle_index,:,:] = cm.hsv(new_theta / (2*np.pi))[:,:-1]
                        scalar_field_for_testing['angle'][triangle_index,:] = new_theta.detach()
                
                # the crucial part
                if zero_test==True:
                    f[triangle_index, :, :] +=  B(r).unsqueeze(-1)*(  + self.V[verts[i]] )
                
                else:

                    #f[triangle_index, :, :] +=  B(r).unsqueeze(-1)*( self.mlps[verts[i]](onering_x)*(-1*B(r).unsqueeze(-1)+1) + self.V[verts[i]] )
                    
                    #f[triangle_index, :, :] +=  B(r).unsqueeze(-1)*( self.mlps[verts[i]](onering_x)@self.rotations[verts[i]] + self.V[verts[i]] )
                    #f[triangle_index, :, :] +=  B(r).unsqueeze(-1)*( self.mlps[verts[i]](onering_x) + self.V[verts[i]] )

                    f[triangle_index, :, :] +=  B(r).unsqueeze(-1)*( self.mlps[verts[i]](posenc(onering_x))@self.rotations[verts[i]] + self.V[verts[i]] )

                ### for POU
                #f[triangle_index,:,:] += self.V[verts[i]]/3
                #f[triangle_index,:,:] += -1*B(r).unsqueeze(-1)*( self.V[verts[0]]+self.V[verts[1]]+self.V[verts[2]] )/9
                blendsum += B(r).unsqueeze(-1)

            f[triangle_index, :, :] += (-1*blendsum + 1) * virtual_vertex

        if return_scalar_fields:    
            return f, scalar_field_for_testing

        return f
        







In [ ]:
def show_f(f, colors):
    # Assuming target_train is already defined (Shape: [num_triangles, 4330, 3])
    num_triangles = f.shape[0]
    
    # Flatten points and colors to create a single point cloud
    points = f.reshape(-1, 3)  # Shape: e.g. [(80 * 4330), 3]
    
    # Create Open3D PointCloud object
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)   # Assign points
    #pcd.colors = o3d.utility.Vector3dVector(np.random.uniform(size=points_np.shape))   # Assign per-triangle colors
    if colors is not None:
        pcd.colors = o3d.utility.Vector3dVector(colors)   # Assign per-triangle colors
    
    # Visualize the point cloud
    o3d.visualization.draw_geometries([pcd], window_name="Colored 3D Point Cloud")
    #o3d.visualization.draw_geometries([pcd, he_mesh], window_name="Colored 3D Point Cloud")

    #o3d.visualization.draw([pcd])
    #o3d.visualization.draw([pcd, he_mesh])
    
def show_stuff(output, target, settings=[]):

    out = output_train.detach().reshape(-1,3)
    target = target_train.reshape(-1,3)

    if 'error' in settings:
    
        error_color  = cm.Reds( 500*np.abs((out - target)**2).sum(-1) )[:,:-1]
        colors=error_color
        #colors = np.tile( error_color , (target_train.shape[0], 1)  )
        print(colors.shape)
        show_f(out.detach().numpy(), colors)
        show_f(target, colors)
        
    if 'bary' in settings:
    
        colors = bary.transpose(0, 2, 1).reshape(-1, 3)
        
        show_f(out.detach().numpy(), colors)
        show_f(target, colors)
        
    if 'blend' in settings:
        colors = scalar_field_for_testing['blend'].reshape(-1,3)
        show_f(out.detach().numpy(), colors)
        #show_f(embedded_triangles.detach().numpy(), colors)
        show_f(target.detach().numpy(), colors)

    if settings == []:
        show_f(out.detach().numpy(), None)
        #show_f(embedded_triangles.detach().numpy(), None)
        show_f(target, None)
    
    '''
    colors = np.tile(bary, (target_train.shape[0], 1))
    show_f(embedded_triangles.detach().numpy(), colors)
    show_f(target_train, colors)
    show_f(output_train.detach().numpy(), colors)
    '''
    if 'angle' in settings:
    
        colors = scalar_field_for_testing['angle'].reshape(-1,3)
        show_f(out.detach().numpy(), colors)
        #show_f(embedded_triangles.detach().numpy(), colors)

    

In [ ]:
'''
#Plotting Blend Functions

x=torch.arange(0,1,0.001)
for v in [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    #plt.plot(x, B(x,v=v))
    #plt.plot(x,B(1-x,v=v))
    plt.plot(x, B(x,v=v)+B(1-x,v=v), label=v)
plt.legend()
plt.show()

x=torch.arange(0,1,0.001)
for v in [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    #plt.plot(x, B(x,v=v))
    #plt.plot(x,B(1-x,v=v))
    plt.plot(x, B(x,v=v), label=v)
plt.legend()
plt.show()
'''


In [ ]:
### Defining the Blended Neural Surface (BNS)
'''
#analytic target shape test

filename = 'data/surfaces/two_triangles.off'
coarse_mesh_o3d = o3d.io.read_triangle_mesh(filename)
coarse_he_mesh = o3d.geometry.HalfEdgeTriangleMesh.create_from_triangle_mesh(coarse_mesh_o3d)
#coarse_he_mesh = o3d.geometry.HalfEdgeTriangleMesh.create_from_triangle_mesh(o3d.geometry.TriangleMesh.create_tetrahedron())
#coarse_he_mesh = o3d.geometry.HalfEdgeTriangleMesh.create_from_triangle_mesh(o3d.geometry.TriangleMesh.create_icosahedron())
#coarse_he_mesh = o3d.geometry.HalfEdgeTriangleMesh.create_from_triangle_mesh(o3d.geometry.TriangleMesh.create_octahedron())

n_faces = np.asarray(coarse_he_mesh.triangles.shape[0])
t_train = torch.rand(10000, 2)
mask = ( t_train[:,1] <= np.sqrt(3)*t_train[:,0] )*( t_train[:,1] <= np.sqrt(3)*(1-t_train[:,0]) ) 
t_train = t_train[mask,:]
t_train = t_train.unsqueeze(0).repeat(n_faces,1,1)

target_train, embedded_triangles, bary = bns.compute_samples_sphere(t_train)

bns = BNS(layer_sizes, from_file=False, shapename= None)

'''

#non-analytic shape test

bns = BNS(layer_sizes, from_file=True, shapename= shapename)

t_train, target_train, target_normals, bary, embedded_triangles = bns.compute_samples(num_samples=1000000)






In [98]:
bns.load_mlps('models/star5_4patch.pth')
bns.load_mlps('models/igea_normalsreg.pth')



import visuals
import importlib
importlib.reload(visuals)
from visuals import *


import differential
import importlib
importlib.reload(differential)
from differential import *


bns.shapename = 'temp'#'igea11706_6patch'
bns_visualiser = BNS_visualiser(bns, mesh_res=7)

settings = ['default', 'angle', 'normals']

bns_visualiser.compute_quantities(settings=settings)
bns_visualiser.show_bns(settings=settings)




/var/folders/3s/dfq9yqvd6y9303d_mb00rpxw0000gp/T/ipykernel_1311/2808364428.py:213: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename)


Computing Normals for Patch 1
Computing Normals for Patch 2
Computing Normals for Patch 3
Computing Normals for Patch 4
Computing Normals for Patch 5
Computing Normals for Patch 6
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.


In [ ]:
output_train, scalar_field_for_testing  = bns.forward(t_train, zero_test=True, return_scalar_fields=True)


In [ ]:
show_stuff(output_train, target_train, settings=[ 'blend'])

In [ ]:
epoch=0

losses=[]
epochs=[]
times=[]
num_patches = bns.num_patches

loss=99999

optimizer = optim.Adam(bns.parameters(), lr=0.001)

In [ ]:




def train_batch_loop(lr=0.01, batch_size=2000, min_loss=0.0005, normals_reg_coeff = 0.0):
    global optimizer, losses, times, epoch, loss, num_patches

    # Update optimizer if learning rate changes
    for g in optimizer.param_groups:
        g['lr'] = lr

    seqs = t_train.shape[0]
    total_points = t_train.shape[1]



    while loss > min_loss:
        time_A = time.time()
        perm = torch.randperm(total_points)
        epoch_loss = 0.0

        for i in range(0, total_points, batch_size):

            time_batchstart = time.time()
            
            idx = perm[i:i + batch_size]  # batch indices
            batch_t = t_train[:, idx, :]
            batch_target = target_train[:, idx, :]
            batch_target_normals = target_normals[:, idx, :]

            

            optimizer.zero_grad()

            if normals_reg_coeff > 0.0:

                #patch_batches = [ batch_t[j,:,:].clone().detach().requires_grad_() for j in range(num_patches) ]
                #for patch_batch in patch_batches:
                #    patch_batch.requires_grad=True
                    
                #output_train, scalar_field_for_testing = bns(torch.stack(patch_batches))
    
                #print('out train', output_train.shape)
                #print('batch t', batch_t.shape)

                batch_t.requires_grad=True
    
                
                
                output_train = bns(batch_t)
                  
                output_normals=diffmod.compute_normals(out=output_train, wrt=batch_t)
                
                normals_reg = (output_normals - batch_target_normals).pow(2).sum(-1).mean()
                
            else:
                output_train = bns(batch_t)
                normals_reg=0.0
            


            position_loss = (output_train - batch_target).pow(2).sum(-1).mean()
            
            
            batch_loss = position_loss + normals_reg_coeff * normals_reg
            batch_loss.backward()
            optimizer.step()

            epoch_loss += batch_loss.item() * batch_t.shape[1]

            time_batchend = time.time()
            this_time = time_batchend - time_batchstart
            if i%100==0:
                print(f"Batch {i}, Loss: {batch_loss:.8f}, Pos. Loss: {position_loss:.8f}, Normals Reg: {(normals_reg_coeff * normals_reg):.8f}, Batch Time: {this_time:.4f}s")

        epoch_loss /= total_points
        loss = torch.tensor(epoch_loss)

        time_B = time.time()
        this_time = time_B - time_A
        times.append(this_time)
        losses.append(loss.detach())

        if epoch % 1 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.8f}, Epoch Time: {this_time:.4f}s")

        if epoch % 1 == 0:
            bns.save_mlps('models/temp.pth')
            print('saved checkpoint')
            plt.plot(np.log10(losses))
            plt.title('log10 of loss')
            plt.show()
            plt.plot(times)
            plt.title('iter time')
            plt.show()

        epoch += 1


In [ ]:
train_batch_loop(lr=0.01, batch_size=1000, min_loss=0.005, normals_reg_coeff = 0.0)


In [84]:
train_batch_loop(lr=0.001, batch_size=10000, min_loss=0.0, normals_reg_coeff = 0.0)

Batch 0, Loss: 0.00019756, Pos. Loss: 0.00019756, Normals Reg: 0.00000000, Batch Time: 0.7648s
Batch 10000, Loss: 0.00019271, Pos. Loss: 0.00019271, Normals Reg: 0.00000000, Batch Time: 0.7433s
Batch 20000, Loss: 0.00018715, Pos. Loss: 0.00018715, Normals Reg: 0.00000000, Batch Time: 0.7364s
Batch 30000, Loss: 0.00018414, Pos. Loss: 0.00018414, Normals Reg: 0.00000000, Batch Time: 0.7407s
Batch 40000, Loss: 0.00017950, Pos. Loss: 0.00017950, Normals Reg: 0.00000000, Batch Time: 0.7463s
Batch 50000, Loss: 0.00017694, Pos. Loss: 0.00017694, Normals Reg: 0.00000000, Batch Time: 0.7361s
Batch 60000, Loss: 0.00017499, Pos. Loss: 0.00017499, Normals Reg: 0.00000000, Batch Time: 0.7449s
Batch 70000, Loss: 0.00017379, Pos. Loss: 0.00017379, Normals Reg: 0.00000000, Batch Time: 0.7406s
Batch 80000, Loss: 0.00017027, Pos. Loss: 0.00017027, Normals Reg: 0.00000000, Batch Time: 0.7412s
Batch 90000, Loss: 0.00017087, Pos. Loss: 0.00017087, Normals Reg: 0.00000000, Batch Time: 0.7351s
Batch 100000, 

KeyboardInterrupt: 

In [70]:
train_batch_loop(lr=0.001, batch_size=10000, min_loss=0.0, normals_reg_coeff = 0.01)


Batch 0, Loss: 0.00572603, Pos. Loss: 0.00182545, Normals Reg: 0.00390058, Batch Time: 4.0393s
Batch 10000, Loss: 0.00515441, Pos. Loss: 0.00174730, Normals Reg: 0.00340712, Batch Time: 4.0595s
Batch 20000, Loss: 0.00453409, Pos. Loss: 0.00161794, Normals Reg: 0.00291616, Batch Time: 4.0575s
Batch 30000, Loss: 0.00385874, Pos. Loss: 0.00144812, Normals Reg: 0.00241063, Batch Time: 4.0212s
Batch 40000, Loss: 0.00331053, Pos. Loss: 0.00125915, Normals Reg: 0.00205138, Batch Time: 4.0323s
Batch 50000, Loss: 0.00291395, Pos. Loss: 0.00108932, Normals Reg: 0.00182463, Batch Time: 4.0392s
Batch 60000, Loss: 0.00254573, Pos. Loss: 0.00092829, Normals Reg: 0.00161744, Batch Time: 4.0161s
Batch 70000, Loss: 0.00232075, Pos. Loss: 0.00079645, Normals Reg: 0.00152431, Batch Time: 4.0247s


KeyboardInterrupt: 

In [ ]:
import visuals
import importlib
importlib.reload(visuals)
from visuals import *
bns.shapename = 'igea11706_6patch'

bns_visualiser = BNS_visualiser(bns, mesh_res=8, zero_test=False)

bns_visualiser.compute_quantities(settings=['default','normals'])
bns_visualiser.show_bns(settings=['default','normals'])

In [ ]:
N.shape

In [ ]:
target_normals.shape

In [ ]:
bns.reset()
show_stuff(output_train, target_train, ['error'])
show_stuff(output_train, target_train, ['blend'])

In [ ]:
#bns = blended_mlp

In [ ]:
########### Translation Equivariance ############
bns.reset()

output_train, scalar_field_for_testing  = bns.forward(t_train)
show_stuff()

bns.V = blended_mlp.V + 2.0
bns.compute_fixed_rotations()

output_train, scalar_field_for_testing  = bns.forward(t_train)
show_stuff()
bns.reset()

In [ ]:
############# Rotation Equivariance ############
bns.reset()
output_train, scalar_field_for_testing  = bns.forward(t_train)
show_stuff()
R = torch.tensor([[0.0, 1.0, 0.0 ],
                  [-1.0, 0.0, 0.0 ],
                 [0.0, 0.0, 1.0]])

bns.coarse_points = bns.coarse_points @ R

bns.compute_fixed_rotations()

output_train, scalar_field_for_testing  = bns.forward(t_train)
show_stuff()
bns.reset()

In [ ]:
############# Translating a Single Coarse Vertex ############
bns.reset()

output_train, scalar_field_for_testing  = bns.forward(t_train)
show_stuff()

T = torch.zeros_like(bnsp.V)
T[0,:] = torch.tensor([-0.1, +0.1, 0.0 ])

bns.V = blended_mlp.V + T

bns.compute_fixed_rotations()

output_train, scalar_field_for_testing  = bns.forward(t_train)
show_stuff()
bns.reset()

In [ ]:
bns.save_mlps('models/star5_4patch.pth')

In [ ]:

#bns.load_mlps('models/icosphere_sphere.pth')

In [ ]:
import visuals
import importlib
importlib.reload(visuals)
from visuals import *
bns.shapename = 'igea11706_6patch'
bns_visualiser = BNS_visualiser(bns, mesh_res=4)

bns_visualiser.compute_quantities()
bns_visualiser.show_bns()